In [1]:
import os
import json
import time
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from firecrawl import FirecrawlApp

# 1. Load Environment Variables
load_dotenv(override=True)

# 2. Initialize Firecrawl
app = FirecrawlApp(api_key=os.getenv("FIRECRAWL_API_KEY"))

# 3. Initialize the Groq LLM Client
llm_client = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2, 
    api_key=os.getenv("GROQ_API_KEY")
)

# 4. Generate URLs for the Bhagavad Gita (Chapter 1, Verses 1 to 47)
# You can change the range or chapter number to scrape different parts of the book
target_urls = [f'https://www.holy-bhagavad-gita.org/chapter/1/verse/{i}' for i in range(1, 48)]
print(f"Loaded {len(target_urls)} Gita verses to scrape.")

master_graph_data = []

# 5. The Extraction Prompt (Keeping the same strict Paninian grammar rules)
extraction_prompt = """
You are a strict backend data extraction API. 
Read the following text containing translated Sanskrit verses.
Extract the main event into the exact JSON schema provided.
OUTPUT ONLY VALID JSON. NO PREAMBLE. NO CONVERSATIONAL TEXT. NO MARKDOWN.

SCHEMA:
{
  "source_url": "string",
  "entities": [
    {"name": "string", "type": "string", "karaka_role": "string"}
  ],
  "relation": "string"
}

TEXT TO ANALYZE:
"""

# 6. The Automation Loop
for url in target_urls:
    print(f"\nScraping {url}...")
    
    try:
        # A. Scrape the raw text
        scraped_data = app.scrape_url(url, formats=['markdown'])
        raw_markdown = scraped_data.markdown or ""
        
        # B. Truncate text to stay under Groq's limits
        safe_markdown = raw_markdown[:5000] 
        
        # C. Send the SAFE truncated text to the LLM
        print("Sending snippet to Groq...")
        response = llm_client.invoke(extraction_prompt + safe_markdown)
        
        # D. Convert the LLM text output into a Python dictionary
        raw_output = response.content
        start_index = raw_output.find('{')
        end_index = raw_output.rfind('}') + 1
        
        if start_index != -1 and end_index != 0:
            clean_text = raw_output[start_index:end_index]
            structured_json = json.loads(clean_text)
            master_graph_data.append(structured_json)
            print("Successfully extracted JSON!")
        else:
            raise ValueError("No JSON structure found.")
            
    except Exception as e:
        print(f"Error processing {url}: {e}")

    # 7. Save the output to your NEW file after every chapter
    with open('automated_graph_dat_2.json', 'w') as f:
        json.dump(master_graph_data, f, indent=2)
        print("Progress saved to automated_graph_dat_2.json")

    # 8. Rate Limiting Pause
    print("Pausing for 3 seconds to respect server limits...")
    time.sleep(3)

print("\nBhagavad Gita Extraction Complete! Ready to push to Neo4j.")

c:\Users\shivc\OneDrive - Indian Institute of Technology Bombay\Desktop\DDKGGS\myenv\Lib\site-packages\firecrawl\v2\types.py:988: UserWarning: Field name "json" in "MonitorPageDiff" shadows an attribute in parent "BaseModel"
  class MonitorPageDiff(BaseModel):
c:\Users\shivc\OneDrive - Indian Institute of Technology Bombay\Desktop\DDKGGS\myenv\Lib\site-packages\firecrawl\v2\types.py:1003: UserWarning: Field name "json" in "MonitorPageSnapshot" shadows an attribute in parent "BaseModel"
  class MonitorPageSnapshot(BaseModel):


Loaded 47 Gita verses to scrape.

Scraping https://www.holy-bhagavad-gita.org/chapter/1/verse/1...
Sending snippet to Groq...
Successfully extracted JSON!
Progress saved to automated_graph_dat_2.json
Pausing for 3 seconds to respect server limits...

Scraping https://www.holy-bhagavad-gita.org/chapter/1/verse/2...
Sending snippet to Groq...
Successfully extracted JSON!
Progress saved to automated_graph_dat_2.json
Pausing for 3 seconds to respect server limits...

Scraping https://www.holy-bhagavad-gita.org/chapter/1/verse/3...
Sending snippet to Groq...
Successfully extracted JSON!
Progress saved to automated_graph_dat_2.json
Pausing for 3 seconds to respect server limits...

Scraping https://www.holy-bhagavad-gita.org/chapter/1/verse/4...
Sending snippet to Groq...
Successfully extracted JSON!
Progress saved to automated_graph_dat_2.json
Pausing for 3 seconds to respect server limits...

Scraping https://www.holy-bhagavad-gita.org/chapter/1/verse/5...
Sending snippet to Groq...
Succes